# GreenMetal AI: Data Preprocessing Pipeline
## Cleaning & Feature Engineering for Phytomining ML

**Objective:** Load raw datasets, handle missing values, remove outliers, and engineer features for ML model training.

**Output:** Clean, merged datasets ready for machine learning

---


## Step 1: Import Libraries


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Configuration
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✓ All libraries imported successfully')

✓ All libraries imported successfully


## Step 2: Load Raw Datasets


In [2]:
print('='*80)
print('LOADING RAW DATASETS')
print('='*80)

# Load raw soil data
raw_soil = pd.read_csv('001_raw_soil_geochemistry.csv')
print(f'\n1. Raw Soil Geochemistry Data')
print(f'   Shape: {raw_soil.shape}')
print(f'   Columns: {list(raw_soil.columns)}')
print(f'\n   First 3 rows:')
print(raw_soil.head(3))
print(f'\n   Missing values:\n{raw_soil.isnull().sum()}')
print(f'\n   Data types:\n{raw_soil.dtypes}')

# Load raw plant data
raw_plants = pd.read_csv('002_raw_hyperaccumulator_plants.csv')
print(f'\n\n2. Raw Hyperaccumulator Plants Data')
print(f'   Shape: {raw_plants.shape}')
print(f'   Columns: {list(raw_plants.columns)}')
print(raw_plants)
print(f'\n   Data quality issues:')
print(f'   - Plant_Efficiency > 1.0: {(raw_plants["Plant_Efficiency"] > 1.0).sum()}')
print(f'   - Missing Biomass: {raw_plants["Biomass_kg_per_plant"].isnull().sum()}')

# Load raw environmental data
raw_env = pd.read_csv('003_raw_environmental_conditions.csv')
print(f'\n\n3. Raw Environmental Conditions Data')
print(f'   Shape: {raw_env.shape}')
print(f'   Missing values: {raw_env.isnull().sum().sum()}')

# Load raw contamination risk data
raw_risk = pd.read_csv('004_raw_contamination_risk.csv')
print(f'\n\n4. Raw Contamination Risk Data')
print(f'   Shape: {raw_risk.shape}')
print(f'   Unique values in key columns:')
print(f'   - Mining_Activity_History: {raw_risk["Mining_Activity_History"].unique()}')
print(f'   - Contamination_Source: {raw_risk["Contamination_Source"].unique()}')

LOADING RAW DATASETS

1. Raw Soil Geochemistry Data
   Shape: (600, 19)
   Columns: ['Sample_ID', 'Latitude', 'Longitude', 'Soil_Depth_cm', 'pH', 'Ni_ppm', 'Co_ppm', 'Zn_ppm', 'Cu_ppm', 'Pb_ppm', 'Cd_ppm', 'Mn_ppm', 'Fe_ppm', 'Soil_OM_percent', 'Soil_Moisture_percent', 'Sand_percent', 'Silt_percent', 'Clay_percent', 'CEC_meq_per_100g']

   First 3 rows:
         Sample_ID   Latitude   Longitude  Soil_Depth_cm        pH  \
0  USGS_SOIL_00000  33.988963 -115.032831              5  2.750891   
1  USGS_SOIL_00001  47.817143 -108.563170             20  9.936414   
2  USGS_SOIL_00002  42.567855 -114.556381             15  4.415641   

        Ni_ppm    Co_ppm     Zn_ppm     Cu_ppm      Pb_ppm     Cd_ppm  \
0  1045.171382  2.601122  28.530443  96.781783   45.745415   0.976287   
1   207.644916  1.601586  43.939802  19.869336  106.026458  10.000231   
2    62.538386  1.241675  51.523838   2.005965   27.561990  10.037555   

       Mn_ppm        Fe_ppm  Soil_OM_percent  Soil_Moisture_percent  \

## Step 3: Data Validation & Quality Report


In [3]:
print('='*80)
print('DATA QUALITY ASSESSMENT')
print('='*80)

def generate_quality_report(df, name):
    print(f'\n{name}:')
    print(f'  Total rows: {len(df)}')
    print(f'  Total columns: {len(df.columns)}')
    print(f'  Duplicates: {df.duplicated().sum()}')
    print(f'  Total missing values: {df.isnull().sum().sum()}')
    print(f'  Missing % of data: {(df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100):.2f}%')
    print(f'\n  Missing values by column:')
    for col in df.columns:
        if df[col].isnull().sum() > 0:
            print(f'    - {col}: {df[col].isnull().sum()} ({df[col].isnull().sum()/len(df)*100:.1f}%)')

generate_quality_report(raw_soil, 'SOIL GEOCHEMISTRY')
generate_quality_report(raw_plants, 'PLANT TRAITS')
generate_quality_report(raw_env, 'ENVIRONMENTAL')
generate_quality_report(raw_risk, 'CONTAMINATION RISK')

DATA QUALITY ASSESSMENT

SOIL GEOCHEMISTRY:
  Total rows: 600
  Total columns: 19
  Duplicates: 0
  Total missing values: 30
  Missing % of data: 0.26%

  Missing values by column:
    - pH: 10 (1.7%)
    - Ni_ppm: 20 (3.3%)

PLANT TRAITS:
  Total rows: 20
  Total columns: 11
  Duplicates: 0
  Total missing values: 1
  Missing % of data: 0.45%

  Missing values by column:
    - Biomass_kg_per_plant: 1 (5.0%)

ENVIRONMENTAL:
  Total rows: 600
  Total columns: 8
  Duplicates: 0
  Total missing values: 20
  Missing % of data: 0.42%

  Missing values by column:
    - Annual_Rainfall_mm: 20 (3.3%)

CONTAMINATION RISK:
  Total rows: 600
  Total columns: 5
  Duplicates: 0
  Total missing values: 196
  Missing % of data: 6.53%

  Missing values by column:
    - Mining_Activity_History: 196 (32.7%)


## Step 4: Clean Soil Geochemistry Data


In [4]:
print('\n' + '='*80)
print('CLEANING SOIL GEOCHEMISTRY DATA')
print('='*80)

soil_clean = raw_soil.copy()

# Step 1: Remove complete duplicates
duplicates_before = soil_clean.duplicated().sum()
soil_clean = soil_clean.drop_duplicates()
print(f'\n1. Removed duplicates: {duplicates_before}')

# Step 2: Impute missing metal concentrations with median
metal_cols = ['Ni_ppm', 'Co_ppm', 'Zn_ppm', 'Cu_ppm', 'Pb_ppm', 'Cd_ppm', 'Mn_ppm', 'Fe_ppm']
imputer_median = SimpleImputer(strategy='median')
soil_clean[metal_cols] = imputer_median.fit_transform(soil_clean[metal_cols])
print(f'\n2. Imputed missing metal concentrations with median')

# Step 3: Impute pH with mean
soil_clean['pH'] = soil_clean['pH'].fillna(soil_clean['pH'].mean())
print(f'   Imputed pH: {soil_clean["pH"].isnull().sum()} remaining nulls')

# Step 4: Impute Soil_OM_percent with mean
soil_clean['Soil_OM_percent'] = soil_clean['Soil_OM_percent'].fillna(soil_clean['Soil_OM_percent'].mean())
print(f'   Imputed Soil_OM_percent: {soil_clean["Soil_OM_percent"].isnull().sum()} remaining nulls')

# Step 5: Clip unrealistic pH values
soil_clean['pH'] = soil_clean['pH'].clip(3.5, 8.5)
print(f'\n3. Clipped pH values to [3.5, 8.5]')

# Step 6: Detect and remove outliers using IQR method
def detect_outliers_iqr(data, column, multiplier=2.5):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - (multiplier * IQR)
    upper_bound = Q3 + (multiplier * IQR)
    return (data[column] < lower_bound) | (data[column] > upper_bound)

before_outlier = len(soil_clean)
outlier_mask = pd.DataFrame(False, index=soil_clean.index, columns=metal_cols)

for col in metal_cols:
    outlier_mask[col] = detect_outliers_iqr(soil_clean, col, multiplier=2.5)

# Mark rows with any outliers
rows_with_outliers = outlier_mask.any(axis=1).sum()
soil_clean = soil_clean[~outlier_mask.any(axis=1)]

print(f'\n4. Outlier removal (IQR method, multiplier=2.5):')
print(f'   Rows with outliers detected: {rows_with_outliers}')
print(f'   Rows removed: {before_outlier - len(soil_clean)}')
print(f'   Remaining rows: {len(soil_clean)}')

# Step 7: Ensure texture adds up to 100%
soil_clean['Texture_Sum'] = soil_clean['Sand_percent'] + soil_clean['Silt_percent'] + soil_clean['Clay_percent']
soil_clean = soil_clean[soil_clean['Texture_Sum'].between(95, 105)]
print(f'\n5. Validated soil texture sums to ~100%')
print(f'   Remaining rows: {len(soil_clean)}')
soil_clean = soil_clean.drop('Texture_Sum', axis=1)

# Step 8: Final validation
print(f'\n6. Final data quality check:')
print(f'   Missing values: {soil_clean.isnull().sum().sum()}')
print(f'   Duplicates: {soil_clean.duplicated().sum()}')
print(f'   Final shape: {soil_clean.shape}')

print(f'\n✓ Soil data cleaning complete!')


CLEANING SOIL GEOCHEMISTRY DATA

1. Removed duplicates: 0

2. Imputed missing metal concentrations with median
   Imputed pH: 0 remaining nulls
   Imputed Soil_OM_percent: 0 remaining nulls

3. Clipped pH values to [3.5, 8.5]

4. Outlier removal (IQR method, multiplier=2.5):
   Rows with outliers detected: 278
   Rows removed: 278
   Remaining rows: 322

5. Validated soil texture sums to ~100%
   Remaining rows: 84

6. Final data quality check:
   Missing values: 0
   Duplicates: 0
   Final shape: (84, 19)

✓ Soil data cleaning complete!


## Step 5: Clean Plant Hyperaccumulator Data


In [5]:
print('\n' + '='*80)
print('CLEANING PLANT HYPERACCUMULATOR DATA')
print('='*80)

plants_clean = raw_plants.copy()

# Fix invalid Plant_Efficiency > 1.0
print(f'\n1. Fixing invalid Plant_Efficiency values')
invalid_eff = (plants_clean['Plant_Efficiency'] > 1.0).sum()
print(f'   Values > 1.0: {invalid_eff}')
plants_clean['Plant_Efficiency'] = plants_clean['Plant_Efficiency'].clip(0.0, 0.99)
print(f'   ✓ Clipped to [0.0, 0.99]')

# Impute missing Biomass_kg_per_plant
print(f'\n2. Imputing missing Biomass values')
missing_biomass = plants_clean['Biomass_kg_per_plant'].isnull().sum()
print(f'   Missing values: {missing_biomass}')
plants_clean['Biomass_kg_per_plant'] = plants_clean['Biomass_kg_per_plant'].fillna(
    plants_clean['Biomass_kg_per_plant'].median()
)
print(f'   ✓ Imputed with median: {plants_clean["Biomass_kg_per_plant"].median():.3f}')

# Validate pH ranges (min < max)
print(f'\n3. Validating pH ranges')
invalid_ph_range = (plants_clean['Min_pH_Tolerance'] >= plants_clean['Max_pH_Tolerance']).sum()
print(f'   Invalid ranges (min >= max): {invalid_ph_range}')

# Validate Max_Accumulation_ppm > 0
print(f'\n4. Validating Max_Accumulation_ppm')
invalid_accum = (plants_clean['Max_Accumulation_ppm'] <= 0).sum()
print(f'   Invalid values (<= 0): {invalid_accum}')

print(f'\n5. Final validation:')
print(f'   Missing values: {plants_clean.isnull().sum().sum()}')
print(f'   Duplicates: {plants_clean.duplicated().sum()}')
print(f'   Final shape: {plants_clean.shape}')

print(f'\n✓ Plant data cleaning complete!')
print(f'\nCleaned plants summary:')
print(plants_clean[['Species', 'Target_Metal', 'Plant_Efficiency', 'Biomass_kg_per_plant']])


CLEANING PLANT HYPERACCUMULATOR DATA

1. Fixing invalid Plant_Efficiency values
   Values > 1.0: 1
   ✓ Clipped to [0.0, 0.99]

2. Imputing missing Biomass values
   Missing values: 1
   ✓ Imputed with median: 0.700

3. Validating pH ranges
   Invalid ranges (min >= max): 0

4. Validating Max_Accumulation_ppm
   Invalid values (<= 0): 0

5. Final validation:
   Missing values: 0
   Duplicates: 0
   Final shape: (20, 11)

✓ Plant data cleaning complete!

Cleaned plants summary:
                       Species Target_Metal  Plant_Efficiency  \
0               Alyssum murale           Ni              0.92   
1              Berkheya coddii           Ni              0.88   
2              Silene vulgaris           Zn              0.85   
3          Arabidopsis halleri           Zn              0.87   
4               Pteris vittata           As              0.78   
5         Thlaspi caerulescens           Zn              0.84   
6              Brassica juncea           Pb              0.80 

## Step 6: Clean Environmental & Risk Data


In [6]:
print('\n' + '='*80)
print('CLEANING ENVIRONMENTAL & CONTAMINATION DATA')
print('='*80)

env_clean = raw_env.copy()
risk_clean = raw_risk.copy()

# Clean environmental data
print(f'\n1. Environmental Data Cleaning')
env_clean['Annual_Rainfall_mm'] = env_clean['Annual_Rainfall_mm'].fillna(env_clean['Annual_Rainfall_mm'].median())
env_clean['Average_Temp_C'] = env_clean['Average_Temp_C'].clip(-10, 50)  # Realistic temperature range
env_clean['Growing_Season_days'] = env_clean['Growing_Season_days'].clip(50, 365)  # Valid growing season
env_clean['Elevation_m'] = env_clean['Elevation_m'].clip(0, 5000)  # Reasonable elevation
print(f'   ✓ Imputed missing rainfall: {env_clean["Annual_Rainfall_mm"].isnull().sum()} nulls')
print(f'   ✓ Clipped temperature, elevation, growing season')
print(f'   Final shape: {env_clean.shape}')

# Clean risk data
print(f'\n2. Contamination Risk Data Cleaning')
risk_clean['Industrial_Proximity_km'] = risk_clean['Industrial_Proximity_km'].clip(0, 50)
risk_clean['Pollutant_Concentration_Index'] = risk_clean['Pollutant_Concentration_Index'].clip(0, 100)
print(f'   ✓ Clipped proximity and concentration index')
print(f'   Unique values:')
print(f'   - Mining_Activity_History: {risk_clean["Mining_Activity_History"].unique()}')
print(f'   - Contamination_Source: {risk_clean["Contamination_Source"].unique()}')
print(f'   Final shape: {risk_clean.shape}')

print(f'\n✓ Environmental & Risk data cleaning complete!')


CLEANING ENVIRONMENTAL & CONTAMINATION DATA

1. Environmental Data Cleaning
   ✓ Imputed missing rainfall: 0 nulls
   ✓ Clipped temperature, elevation, growing season
   Final shape: (600, 8)

2. Contamination Risk Data Cleaning
   ✓ Clipped proximity and concentration index
   Unique values:
   - Mining_Activity_History: ['Current' nan 'Past']
   - Contamination_Source: ['Mining' 'Unknown' 'Urban' 'Agriculture' 'Industrial']
   Final shape: (600, 5)

✓ Environmental & Risk data cleaning complete!


## Step 7: Merge All Datasets


In [8]:
print('\n' + '='*80)
print('MERGING DATASETS')
print('='*80)

# Only keep rows that exist in all datasets
common_sample_ids = pd.Index(soil_clean['Sample_ID']).intersection(env_clean['Sample_ID']).intersection(risk_clean['Sample_ID'])
print(f'\nCommon Sample IDs across all datasets: {len(common_sample_ids)}')

# Filter all datasets to common samples
soil_filtered = soil_clean[soil_clean['Sample_ID'].isin(common_sample_ids)]
env_filtered = env_clean[env_clean['Sample_ID'].isin(common_sample_ids)]
risk_filtered = risk_clean[risk_clean['Sample_ID'].isin(common_sample_ids)]

print(f'Filtered dataset sizes:')
print(f'  Soil: {len(soil_filtered)}')
print(f'  Environmental: {len(env_filtered)}')
print(f'  Risk: {len(risk_filtered)}')

# Merge datasets
merged_df = soil_filtered.merge(env_filtered[['Sample_ID', 'Annual_Rainfall_mm', 'Average_Temp_C', 'Growing_Season_days', 'Elevation_m', 'Solar_Radiation_MJ_m2', 'Wind_Speed_m_s', 'Soil_Drainage']],
                                on='Sample_ID', how='left')
merged_df = merged_df.merge(risk_filtered[['Sample_ID', 'Industrial_Proximity_km', 'Mining_Activity_History', 'Contamination_Source', 'Pollutant_Concentration_Index']],
                           on='Sample_ID', how='left')

print(f'\nFinal merged dataset:')
print(f'  Shape: {merged_df.shape}')
print(f'  Columns: {len(merged_df.columns)}')
print(f'  Missing values: {merged_df.isnull().sum().sum()}')

print(f'\n✓ Dataset merge complete!')


MERGING DATASETS

Common Sample IDs across all datasets: 84
Filtered dataset sizes:
  Soil: 84
  Environmental: 84
  Risk: 84

Final merged dataset:
  Shape: (84, 30)
  Columns: 30
  Missing values: 31

✓ Dataset merge complete!


## Step 8: Feature Engineering


In [9]:
print('\n' + '='*80)
print('FEATURE ENGINEERING')
print('='*80)

processed_df = merged_df.copy()

# 1. Average metal concentration
metal_cols = ['Ni_ppm', 'Co_ppm', 'Zn_ppm', 'Cu_ppm', 'Pb_ppm', 'Cd_ppm', 'Mn_ppm', 'Fe_ppm']
processed_df['Avg_Metal_ppm'] = processed_df[metal_cols].mean(axis=1)
processed_df['Total_Metal_ppm'] = processed_df[metal_cols].sum(axis=1)
processed_df['Max_Metal_ppm'] = processed_df[metal_cols].max(axis=1)
print(f'\n1. Metal concentration aggregate features:')
print(f'   - Avg_Metal_ppm')
print(f'   - Total_Metal_ppm')
print(f'   - Max_Metal_ppm')

# 2. Soil texture classification
def classify_soil_texture(sand, silt, clay):
    if sand > 70:
        return 'Sandy'
    elif clay > 40:
        return 'Clayey'
    elif silt > 50:
        return 'Silty'
    else:
        return 'Loam'

processed_df['Soil_Texture_Type'] = processed_df.apply(
    lambda row: classify_soil_texture(row['Sand_percent'], row['Silt_percent'], row['Clay_percent']),
    axis=1
)
print(f'\n2. Soil texture classification:')
print(f'   {processed_df["Soil_Texture_Type"].value_counts().to_dict()}')

# 3. Categorical encoding for Soil_Drainage
drainage_mapping = {'Poor': 1, 'Moderate': 2, 'Well': 3}
processed_df['Drainage_Score'] = processed_df['Soil_Drainage'].map(drainage_mapping)
print(f'\n3. Drainage encoding: {drainage_mapping}')

# 4. Mining activity encoding
mining_mapping = {'None': 0, 'Past': 1, 'Current': 2}
processed_df['Mining_Activity_Score'] = processed_df['Mining_Activity_History'].map(mining_mapping)
print(f'\n4. Mining activity encoding: {mining_mapping}')

# 5. Contamination source encoding
contamination_mapping = {'Mining': 4, 'Industrial': 3, 'Agriculture': 2, 'Urban': 1, 'Unknown': 0}
processed_df['Contamination_Source_Score'] = processed_df['Contamination_Source'].map(contamination_mapping)
print(f'\n5. Contamination source encoding: {contamination_mapping}')

# 6. Environmental favorability score
processed_df['Rainfall_Normalized'] = (processed_df['Annual_Rainfall_mm'] - processed_df['Annual_Rainfall_mm'].min()) / (processed_df['Annual_Rainfall_mm'].max() - processed_df['Annual_Rainfall_mm'].min())
processed_df['Temp_Normalized'] = (processed_df['Average_Temp_C'] - processed_df['Average_Temp_C'].min()) / (processed_df['Average_Temp_C'].max() - processed_df['Average_Temp_C'].min())
processed_df['Environmental_Score'] = (processed_df['Rainfall_Normalized'] + processed_df['Temp_Normalized'] + processed_df['Drainage_Score']/3) / 3
print(f'\n6. Environmental favorability score created')

# 7. Contamination viability score
processed_df['Contamination_Viability'] = (processed_df['Pollutant_Concentration_Index'] / 100) * (1 - processed_df['Industrial_Proximity_km'] / 50).clip(0, 1)
print(f'\n7. Contamination viability score created')

print(f'\n✓ Feature engineering complete!')
print(f'\nNew features created: 14')
print(f'Total columns: {len(processed_df.columns)}')


FEATURE ENGINEERING

1. Metal concentration aggregate features:
   - Avg_Metal_ppm
   - Total_Metal_ppm
   - Max_Metal_ppm

2. Soil texture classification:
   {'Loam': 80, 'Sandy': 2, 'Clayey': 1, 'Silty': 1}

3. Drainage encoding: {'Poor': 1, 'Moderate': 2, 'Well': 3}

4. Mining activity encoding: {'None': 0, 'Past': 1, 'Current': 2}

5. Contamination source encoding: {'Mining': 4, 'Industrial': 3, 'Agriculture': 2, 'Urban': 1, 'Unknown': 0}

6. Environmental favorability score created

7. Contamination viability score created

✓ Feature engineering complete!

New features created: 14
Total columns: 41


## Step 9: Export Cleaned Datasets


In [10]:
print('\n' + '='*80)
print('EXPORTING CLEANED DATASETS')
print('='*80)

# Export individual cleaned datasets
soil_clean.to_csv('101_cleaned_soil_geochemistry.csv', index=False)
print(f'\n✓ Saved: 101_cleaned_soil_geochemistry.csv ({len(soil_clean)} rows)')

plants_clean.to_csv('102_cleaned_hyperaccumulator_plants.csv', index=False)
print(f'✓ Saved: 102_cleaned_hyperaccumulator_plants.csv ({len(plants_clean)} rows)')

env_clean.to_csv('103_cleaned_environmental_conditions.csv', index=False)
print(f'✓ Saved: 103_cleaned_environmental_conditions.csv ({len(env_clean)} rows)')

risk_clean.to_csv('104_cleaned_contamination_risk.csv', index=False)
print(f'✓ Saved: 104_cleaned_contamination_risk.csv ({len(risk_clean)} rows)')

# Export merged and processed dataset
processed_df.to_csv('200_master_processed_dataset.csv', index=False)
print(f'\n✓ Saved: 200_master_processed_dataset.csv ({len(processed_df)} rows, {len(processed_df.columns)} columns)')

print(f'\n✓ All datasets exported successfully!')


EXPORTING CLEANED DATASETS

✓ Saved: 101_cleaned_soil_geochemistry.csv (84 rows)
✓ Saved: 102_cleaned_hyperaccumulator_plants.csv (20 rows)
✓ Saved: 103_cleaned_environmental_conditions.csv (600 rows)
✓ Saved: 104_cleaned_contamination_risk.csv (600 rows)

✓ Saved: 200_master_processed_dataset.csv (84 rows, 41 columns)

✓ All datasets exported successfully!


## Step 10: Data Summary & Statistics


In [12]:
print('\n' + '='*80)
print('FINAL DATA SUMMARY & STATISTICS')
print('='*80)

print(f'\n1. PROCESSED DATASET OVERVIEW')
print(f'   Total rows: {len(processed_df)}')
print(f'   Total columns: {len(processed_df.columns)}')
print(f'   Memory usage: {processed_df.memory_usage(deep=True).sum() / 1024:.2f} KB')

print(f'\n2. SOIL GEOCHEMISTRY STATISTICS')
print(f'   pH range: [{processed_df["pH"].min():.2f}, {processed_df["pH"].max():.2f}]')
print(f'   Ni_ppm range: [{processed_df["Ni_ppm"].min():.2f}, {processed_df["Ni_ppm"].max():.2f}]')
print(f'   Zn_ppm range: [{processed_df["Zn_ppm"].min():.2f}, {processed_df["Zn_ppm"].max():.2f}]')
print(f'   Soil_OM_percent range: [{processed_df["Soil_OM_percent"].min():.2f}, {processed_df["Soil_OM_percent"].max():.2f}]')

print(f'\n3. METAL CONCENTRATION SUMMARY')
print(f'   Avg_Metal_ppm: {processed_df["Avg_Metal_ppm"].describe().to_dict()}')
print(f'   Total_Metal_ppm range: [{processed_df["Total_Metal_ppm"].min():.2f}, {processed_df["Total_Metal_ppm"].max():.2f}]')

print(f'\n4. ENVIRONMENTAL CONDITIONS')
print(f'   Annual_Rainfall_mm range: [{processed_df["Annual_Rainfall_mm"].min():.2f}, {processed_df["Annual_Rainfall_mm"].max():.2f}]')
print(f'   Average_Temp_C range: [{processed_df["Average_Temp_C"].min():.2f}, {processed_df["Average_Temp_C"].max():.2f}]')

print(f'\n5. CATEGORICAL DISTRIBUTION')
print(f'   Soil Texture Types: {dict(processed_df["Soil_Texture_Type"].value_counts())}')
print(f'   Drainage Scores: {dict(processed_df["Drainage_Score"].value_counts().sort_index())}')
print(f'   Mining Activity Scores: {dict(processed_df["Mining_Activity_Score"].value_counts().sort_index())}')

print(f'\n6. DATA QUALITY FINAL CHECK')
print(f'   Missing values: {processed_df.isnull().sum().sum()}')
print(f'   Duplicates: {processed_df.duplicated().sum()}')
print(f'   Infinity values: {np.isinf(processed_df.select_dtypes(np.number)).sum().sum()}')

print(f'\n✓ DATA PREPROCESSING COMPLETE!')
print(f'\nReady for model training!')


FINAL DATA SUMMARY & STATISTICS

1. PROCESSED DATASET OVERVIEW
   Total rows: 84
   Total columns: 41
   Memory usage: 46.23 KB

2. SOIL GEOCHEMISTRY STATISTICS
   pH range: [3.50, 8.50]
   Ni_ppm range: [0.57, 529.25]
   Zn_ppm range: [2.46, 1918.00]
   Soil_OM_percent range: [2.70, 9.66]

3. METAL CONCENTRATION SUMMARY
   Avg_Metal_ppm: {'count': 84.0, 'mean': 737.2433186488694, 'std': 498.82631957849804, 'min': 61.04109984125884, '25%': 388.66377878931814, '50%': 565.3620997816413, '75%': 932.2371772861409, 'max': 2247.1762283970797}
   Total_Metal_ppm range: [488.33, 17977.41]

4. ENVIRONMENTAL CONDITIONS
   Annual_Rainfall_mm range: [102.63, 1543.87]
   Average_Temp_C range: [-2.47, 33.05]

5. CATEGORICAL DISTRIBUTION
   Soil Texture Types: {'Loam': np.int64(80), 'Sandy': np.int64(2), 'Clayey': np.int64(1), 'Silty': np.int64(1)}
   Drainage Scores: {1: np.int64(33), 2: np.int64(19), 3: np.int64(32)}
   Mining Activity Scores: {1.0: np.int64(27), 2.0: np.int64(26)}

6. DATA QUALIT

### Saving the Final Processed Dataset

In [14]:
print('\n' + '='*80)
print('EXPORTING FINAL DATASET')
print('='*80)

processed_df.to_csv('final_dataset.csv', index=False)
print(f'\n✓ Saved: final_dataset.csv ({len(processed_df)} rows, {len(processed_df.columns)} columns)')

print(f'\nRemember that `102_cleaned_hyperaccumulator_plants.csv` is also available as a separate cleaned dataset.')


EXPORTING FINAL DATASET

✓ Saved: final_dataset.csv (84 rows, 41 columns)

Remember that `102_cleaned_hyperaccumulator_plants.csv` is also available as a separate cleaned dataset.
